# Notebook 01 — Extracción inicial de vídeos de YouTube sobre pesca en Valmayor

Este notebook implementa la primera fase del pipeline del TFG: la **recuperación amplia de vídeos de YouTube** potencialmente relacionados con la actividad de pesca en **Valmayor**. En esta etapa se prioriza la **cobertura** frente al filtrado estricto, con el objetivo de construir un conjunto candidato suficientemente amplio para las fases posteriores.

A diferencia de versiones previas más centradas en `carpfishing`, en esta iteración se amplía el scraping para cubrir distintas modalidades y especies de pesca deportiva presentes en vídeos sobre Valmayor, incluyendo carpa, black bass, lucio y lucioperca, además de consultas más generales de pesca.

El resultado es un dataset **raw** con una fila por vídeo y metadatos básicos reutilizables en los notebooks de preparación, filtrado y extracción con LLM.

## Objetivo

- Recuperar identificadores de vídeos a partir de múltiples consultas temáticas relacionadas con `Valmayor` y la actividad de pesca.
- Cubrir tanto consultas generales como consultas más específicas por especie o modalidad.
- Descargar metadatos por vídeo con `yt-dlp` sin descargar el contenido multimedia.
- Mantener un conjunto amplio de candidatos para no perder muestra en etapas tempranas.
- Normalizar la fecha de publicación y añadir variables temporales útiles.
- Persistir el dataset en formato Parquet y CSV.

## Entradas y salidas

**Entrada:** consultas abiertas a YouTube mediante `scrapetube` y descarga de metadatos con `yt-dlp`.

**Salida:** ficheros `videos_raw.parquet` y `videos_raw.csv` en `data/youtube/raw/`.

## Preparación del entorno

Montaje de Google Drive y definición de la ruta `RAW_DIR`, donde se guardará el dataset bruto con los vídeos recuperados desde YouTube.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

import os
from pathlib import Path

# Carpeta base donde estás trabajando (dentro de PIDS4jj directamente)
BASE_DIR = "/content/drive/MyDrive/PIDS4jjj2"

# En V2 dejamos claro que lo que sale de YouTube va a "raw"
RAW_DIR = Path(BASE_DIR) / "data" / "youtube" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("RAW_DIR:", RAW_DIR)


Mounted at /content/drive
BASE_DIR: /content/drive/MyDrive/PIDS4jjj2
RAW_DIR: /content/drive/MyDrive/PIDS4jjj2/data/youtube/raw


## Búsqueda amplia de vídeos candidatos

En esta fase se utilizan consultas temáticas amplias relacionadas con `Valmayor`, pesca deportiva, especies frecuentes y distintas modalidades de pesca. El objetivo no es filtrar todavía con mucha exhaustividad, sino maximizar el número de vídeos candidatos para que el filtrado semántico y contextual se realice más adelante.

Se instala `scrapetube` para recuperar resultados de búsqueda y `yt-dlp` para descargar metadatos por vídeo.

In [2]:
!pip -q install scrapetube yt-dlp pandas tqdm pyarrow


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 63.4 MB/s eta 0:00:00


In [4]:
import scrapetube
import pandas as pd
from tqdm import tqdm

QUERIES = [
    # -------------------------
    # Bloque A: pesca general en Valmayor
    # -------------------------
    "valmayor pesca",
    "valmayor embalse pesca",
    "pesca valmayor",
    "pesca embalse valmayor",
    "valmayor pesca deportiva",
    "pesca deportiva valmayor",
    "pesca desde orilla valmayor",
    "pesca embarcado valmayor",

    # -------------------------
    # Bloque B: carpa / carpfishing
    # -------------------------
    "valmayor carpa",
    "valmayor carpas",
    "pesca carpa valmayor",
    "carpas embalse valmayor",
    "valmayor carpfishing",
    "carpfishing valmayor",
    "sesion valmayor carpas",
    "valmayor bait boat carpa",
    "valmayor session carp",
    "embalse valmayor carp fishing",
    "pesca nocturna valmayor",
    "carpfishing madrid valmayor",

    # -------------------------
    # Bloque C: black bass
    # -------------------------
    "valmayor black bass",
    "black bass valmayor",
    "pesca black bass valmayor",
    "bass fishing valmayor",
    "valmayor bass fishing",

    # -------------------------
    # Bloque D: lucio
    # -------------------------
    "valmayor lucio",
    "lucio valmayor",
    "pesca lucio valmayor",
    "spinning lucio valmayor",

    # -------------------------
    # Bloque E: lucioperca
    # -------------------------
    "valmayor lucioperca",
    "lucioperca valmayor",
    "pesca lucioperca valmayor",
    "spinning lucioperca valmayor",

    # -------------------------
    # Bloque F: modalidades / técnicas generales
    # -------------------------
    "spinning valmayor",
    "pesca spinning valmayor",
    "feeder valmayor",
    "pesca depredadores valmayor",

    # -------------------------
    # Bloque G: queries temporales
    # -------------------------
    "valmayor pesca 2023",
    "valmayor pesca 2024",
    "valmayor pesca 2025",
    "valmayor carpa 2023",
    "valmayor carpa 2024",
    "valmayor carpa 2025",
]

LIMIT_PER_QUERY = 150

video_ids = []
seen = set()
query_hits = []

for q in QUERIES:
    count_before = len(video_ids)
    results = scrapetube.get_search(q, limit=LIMIT_PER_QUERY)

    for v in results:
        vid = v.get("videoId")
        if vid and vid not in seen:
            seen.add(vid)
            video_ids.append(vid)

    count_after = len(video_ids)
    query_hits.append({
        "query": q,
        "new_ids_added": count_after - count_before,
        "total_unique_so_far": count_after
    })

query_hits_df = pd.DataFrame(query_hits)

print("Número total de queries:", len(QUERIES))
print("Total video_ids únicos:", len(video_ids))

query_hits_df

Número total de queries: 43
Total video_ids únicos: 1397


,query,new_ids_added,total_unique_so_far
0,valmayor pesca,149,149
1,valmayor embalse pesca,44,193
2,pesca valmayor,9,202
3,pesca embalse valmayor,17,219
4,valmayor pesca deportiva,38,257
5,pesca deportiva valmayor,8,265
6,pesca desde orilla valmayor,68,333
7,pesca embarcado valmayor,18,351
8,valmayor carpa,18,369
9,valmayor carpas,4,373


In [5]:
pd.Series(video_ids).head(20)

,0
0,_LMjvsur7Ps
1,pSayYMpES4o
2,wsPZi-pGDoU
3,EoL7-8AOmeg
4,w1dRYWKiQ3s
5,sZudvyjey3Q
6,NhPwkRWBCAw
7,0ahr_d-RK28
8,0rTQOlm2rO8
9,LRRMKjsKvSQ


In [6]:
query_hits_df.sort_values("new_ids_added", ascending=False).reset_index(drop=True)

,query,new_ids_added,total_unique_so_far
0,valmayor pesca,149,149
1,valmayor bait boat carpa,134,600
2,pesca nocturna valmayor,112,808
3,valmayor session carp,82,682
4,feeder valmayor,80,1163
5,valmayor pesca 2023,78,1303
6,pesca desde orilla valmayor,68,333
7,pesca depredadores valmayor,62,1225
8,valmayor black bass,58,872
9,valmayor embalse pesca,44,193


## Extracción y simplificación de metadatos

A partir de los `video_id` candidatos se descargan metadatos con `yt-dlp`. Aunque la respuesta completa contiene muchos campos, en este pipeline solo se conservan aquellos que resultan trazables, explicables y útiles para el resto del proyecto.

Se definen dos funciones auxiliares:

- `fetch_metadata`: descarga el JSON de metadatos del vídeo sin descargar el contenido multimedia.
- `parse_metadata`: convierte ese JSON en un registro plano con una fila por vídeo.

In [7]:
import subprocess
import json
import pandas as pd
from tqdm import tqdm

In [8]:
def fetch_metadata(video_id: str) -> dict:
    """
    Devuelve un diccionario con metadatos del vídeo usando yt-dlp.
    No descarga el vídeo, solo extrae información en formato JSON.
    """

    url = f"https://www.youtube.com/watch?v={video_id}"

    cmd = [
        "yt-dlp",
        "-J",                  # salida JSON
        "--skip-download",     # no descarga vídeo
        "--no-warnings",
        url
    ]

    try:
        out = subprocess.check_output(cmd, stderr=subprocess.STDOUT, text=True)
        data = json.loads(out)
        return data
    except subprocess.CalledProcessError:
        # Si falla (por ejemplo, vídeo borrado o restringido), devolvemos None
        return None

In [9]:
def parse_metadata(meta: dict) -> dict:
    """
    Convierte el JSON devuelto por yt-dlp en un registro plano.
    Solo conserva las columnas más útiles para el pipeline.
    """

    if meta is None:
        return None

    return {
        "video_id": meta.get("id"),
        "title": meta.get("title"),
        "description": meta.get("description"),
        "channel": meta.get("channel"),
        "channel_id": meta.get("channel_id"),
        "uploader": meta.get("uploader"),
        "upload_date": meta.get("upload_date"),
        "timestamp": meta.get("timestamp"),
        "duration": meta.get("duration"),
        "view_count": meta.get("view_count"),
        "like_count": meta.get("like_count"),
        "comment_count": meta.get("comment_count"),
        "tags": meta.get("tags"),
        "webpage_url": meta.get("webpage_url"),
        "language": meta.get("language"),
        "availability": meta.get("availability"),
    }

## Descarga de metadatos y construcción del dataset bruto

Se recorren los identificadores recuperados y, para cada uno, se descargan sus metadatos con `yt-dlp`. Los vídeos cuyo JSON no se puede recuperar se contabilizan aparte. El resultado se concentra en un único `DataFrame` con una fila por vídeo válido.

In [10]:
records = []
failed = 0

for vid in tqdm(video_ids, desc="Descargando metadatos"):
    meta = fetch_metadata(vid)
    row = parse_metadata(meta)

    if row is None:
        failed += 1
        continue

    records.append(row)

videos_df = pd.DataFrame(records).drop_duplicates(subset=["video_id"]).reset_index(drop=True)

print("Videos parseados:", len(videos_df))
print("Fallidos:", failed)
videos_df.head(3)

Descargando metadatos: 100%|██████████| 1397/1397 [50:36<00:00,  2.17s/it]

Videos parseados: 932
Fallidos: 465


,video_id,title,description,channel,channel_id,uploader,upload_date,timestamp,duration,view_count,like_count,comment_count,tags,webpage_url,language,availability
0,_LMjvsur7Ps,¿Está muerto VALMAYOR?🎣,"🎣 ¿Día de pesca en Valmayor?… Bueno, intentarl...",PESCABICHOS,UCKw1WymRHREDpIKJS5gkl_g,PESCABICHOS,20260216,1771281548,514,4094,127.0,49.0,[],https://www.youtube.com/watch?v=_LMjvsur7Ps,es-US,public
1,pSayYMpES4o,04.07.2020 Visita labor AgentesForestalesCM en...,,112cmadrid,UCFjE1iouSSjkFILE4U3WXCg,112cmadrid,20200704,1593854805,81,1525,3.0,1.0,[#ASEM112 #Madrid112 #AgentesForestalesCM],https://www.youtube.com/watch?v=pSayYMpES4o,None,public
2,wsPZi-pGDoU,Embalse de Valmayor. lucioperca 750g,Перший судак на новий спініг 750г,Roman Burbil,UC81epCKhGxMr8m43FkkYM2A,Roman Burbil,20201011,1602459248,28,391,3.0,NaN,[],https://www.youtube.com/watch?v=wsPZi-pGDoU,es,public


In [11]:
videos_df[["video_id", "title", "channel", "upload_date", "view_count"]].sample(
    min(10, len(videos_df)),
    random_state=42
)

,video_id,title,channel,upload_date,view_count
829,dJTDKmWcN_4,2 Dias de Pesca Por Extremadura Oct 2021,JossBassManz,20211030,480
70,5RJUr8RY9pI,Valmayor - 07 Lake Valmayor (Instrumental) Vid...,Valmayor Metal,20141125,107
631,i-LnkEYO7ks,"Autumn 2:3, Carp fishing in The Mountains, Cat...",CatchAdventure_official,20220121,320
506,AJb4vb5tWdQ,III Open Worldpesk La Farola Sierra Brava,Marcos Worldpesk,20140714,597
703,gKk9I4V3714,"Surfcasting 💪🎣 PRIMER RETO CONSEGUIDO, LO NUN...",Surfcasting El Dorado,20210512,8416
96,WVy49PtW3iQ,Pantano de valmayor - ruteando con la biciclet...,nuestraformadevida,20190526,51
465,UwVrCVWNF-Y,"Riverside camping, live bait fishing, trophy h...",Volyn Bushcraft,20260503,115009
86,yL12ArUjzEE,CARPFISHING EN EL EMBALSE DE VALMAYOR | ENERO ...,Manny Carpangler,20250117,763
530,2rMa1K47ViY,Premier poisson de 2022,CARPE STORY,20220320,4388
350,ek_gaZTRqRM,RUTA 3 ( Escorial - Valmayor - Boadilla - Casa...,Me compre una mtb,20201213,243


## Estandarización temporal

Antes de guardar el dataset se normaliza la fecha de publicación y se añaden variables temporales derivadas (`year`, `month`, `quarter`, `year_quarter`). Estas columnas serán útiles en fases posteriores, especialmente si finalmente se construye el dataset de modelado a nivel trimestral.

In [12]:
# Convertimos la fecha de publicación a formato datetime
videos_df["published_at"] = pd.to_datetime(
    videos_df["upload_date"],
    format="%Y%m%d",
    errors="coerce"
)

# Si faltase upload_date, intentamos usar timestamp
mask_missing_date = videos_df["published_at"].isna() & videos_df["timestamp"].notna()
videos_df.loc[mask_missing_date, "published_at"] = pd.to_datetime(
    videos_df.loc[mask_missing_date, "timestamp"],
    unit="s",
    errors="coerce"
)

# Variables temporales derivadas
videos_df["year"] = videos_df["published_at"].dt.year
videos_df["month"] = videos_df["published_at"].dt.month
videos_df["quarter"] = videos_df["published_at"].dt.quarter

videos_df["year_quarter"] = (
    videos_df["year"].astype("Int64").astype(str)
    + "-Q" +
    videos_df["quarter"].astype("Int64").astype(str)
)

videos_df[["video_id", "title", "published_at", "year", "quarter", "year_quarter"]].head(10)

,video_id,title,published_at,year,quarter,year_quarter
0,_LMjvsur7Ps,¿Está muerto VALMAYOR?🎣,2026-02-16,2026,1,2026-Q1
1,pSayYMpES4o,04.07.2020 Visita labor AgentesForestalesCM en...,2020-07-04,2020,3,2020-Q3
2,wsPZi-pGDoU,Embalse de Valmayor. lucioperca 750g,2020-10-11,2020,4,2020-Q4
3,EoL7-8AOmeg,Embalse de Valmayor. lucioperca 2.3 kg,2021-01-23,2021,1,2021-Q1
4,w1dRYWKiQ3s,Lucio de valmayor,2014-11-24,2014,4,2014-Q4
5,sZudvyjey3Q,la carpa de valmayor,2009-01-25,2009,1,2009-Q1
6,NhPwkRWBCAw,PESCA EMBALSE DE VALMAYOR,2020-06-01,2020,2,2020-Q2
7,0ahr_d-RK28,Valmayor LUCIO (SZCZUPAK) 3-part 2,2009-11-16,2009,4,2009-Q4
8,0rTQOlm2rO8,Pesca Carp Embalse de Valmayor,2026-04-06,2026,2,2026-Q2
9,LRRMKjsKvSQ,Valmayor lucio(szczupak) continuacion,2010-06-04,2010,2,2010-Q2


## Persistencia del dataset raw

El dataset bruto se guarda en `data/youtube/raw/` en formato Parquet y CSV. En esta primera fase todavía no se aplica un filtrado semántico estricto, ya que se prioriza la cobertura para no perder muestra demasiado pronto.

In [13]:
from datetime import datetime

# Marca temporal del scraping
videos_df["scraped_at_utc"] = datetime.utcnow().isoformat()

# Reordenamos columnas para dejar el dataset más limpio
preferred_order = [
    "video_id", "title", "description",
    "channel", "channel_id", "uploader",
    "published_at", "year", "month", "quarter", "year_quarter",
    "upload_date", "timestamp", "duration",
    "view_count", "like_count", "comment_count",
    "tags", "language", "availability",
    "webpage_url", "scraped_at_utc"
]

cols_existing = [c for c in preferred_order if c in videos_df.columns] + \
                [c for c in videos_df.columns if c not in preferred_order]

videos_df = videos_df[cols_existing]

# Guardado en disco
raw_parquet_path = RAW_DIR / "videos_raw.parquet"
raw_csv_path = RAW_DIR / "videos_raw.csv"

videos_df.to_parquet(raw_parquet_path, index=False)
videos_df.to_csv(raw_csv_path, index=False)

print("Guardado parquet:", raw_parquet_path)
print("Guardado csv:", raw_csv_path)
print("Shape final:", videos_df.shape)

/tmp/ipykernel_15832/2145607650.py:4: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  videos_df["scraped_at_utc"] = datetime.utcnow().isoformat()


Guardado parquet: /content/drive/MyDrive/PIDS4jjj2/data/youtube/raw/videos_raw.parquet
Guardado csv: /content/drive/MyDrive/PIDS4jjj2/data/youtube/raw/videos_raw.csv
Shape final: (932, 22)


In [14]:
# Comprobación rápida: cuántos vídeos mencionan explícitamente Valmayor
mask_valmayor = (
    videos_df["title"].fillna("").str.contains("valmayor", case=False, na=False) |
    videos_df["description"].fillna("").str.contains("valmayor", case=False, na=False)
)

print("Vídeos con 'valmayor' en title o description:", mask_valmayor.sum())
print("Porcentaje:", round(100 * mask_valmayor.mean(), 2), "%")

# Comprobación rápida: términos de pesca frecuentes
mask_fishing_terms = (
    videos_df["title"].fillna("").str.contains(
        r"carpa|carpas|carpfishing|black bass|bass|lucio|lucioperca|spinning|pesca",
        case=False,
        na=False,
        regex=True
    ) |
    videos_df["description"].fillna("").str.contains(
        r"carpa|carpas|carpfishing|black bass|bass|lucio|lucioperca|spinning|pesca",
        case=False,
        na=False,
        regex=True
    )
)

print("Vídeos con términos generales de pesca/especie:", mask_fishing_terms.sum())
print("Porcentaje:", round(100 * mask_fishing_terms.mean(), 2), "%")

Vídeos con 'valmayor' en title o description: 332
Porcentaje: 35.62 %
Vídeos con términos generales de pesca/especie: 660
Porcentaje: 70.82 %


In [15]:
print("Número de vídeos únicos:", videos_df["video_id"].nunique())
print("Número de canales únicos:", videos_df["channel_id"].nunique())

videos_df[["title", "description"]].head(10)

Número de vídeos únicos: 932
Número de canales únicos: 603


,title,description
0,¿Está muerto VALMAYOR?🎣,"🎣 ¿Día de pesca en Valmayor?… Bueno, intentarl..."
1,04.07.2020 Visita labor AgentesForestalesCM en...,
2,Embalse de Valmayor. lucioperca 750g,Перший судак на новий спініг 750г
3,Embalse de Valmayor. lucioperca 2.3 kg,"судак 2,3кг"
4,Lucio de valmayor,Captura y suelta siempre
5,la carpa de valmayor,esta es una carpa lo bastante grande como para...
6,PESCA EMBALSE DE VALMAYOR,DANY TE ASTEPT PE FACEBOOK-UL MEU PETRICA DUMI...
7,Valmayor LUCIO (SZCZUPAK) 3-part 2,
8,Pesca Carp Embalse de Valmayor,
9,Valmayor lucio(szczupak) continuacion,Otra vez gracias a BIGBAIT nos salvamos de sal...


## Conclusión

Este notebook deja completada la fase de **recuperación amplia de vídeos candidatos** del pipeline. El resultado es un dataset bruto con metadatos de vídeos potencialmente relacionados con `Valmayor`, todavía sin aplicar filtros semánticos o contextuales estrictos.

Esta decisión es deliberada: en esta etapa se prioriza la **cobertura** para evitar perder muestra demasiado pronto. Además, en esta nueva versión el scraping ya no se limita a `carpfishing`, sino que amplía el foco a la **actividad de pesca** en Valmayor, incluyendo distintas especies y modalidades.

El dataset generado aquí servirá como entrada del notebook 02, donde se realizará la preparación inicial del conjunto y se dejarán listas las columnas necesarias para el filtrado y análisis posteriores.